In [ ]:
!git clone https://github.com/marcoshernanz/llm-lab.git /kaggle/working/llm-lab
%cd /kaggle/working/llm-lab
!git pull
!git checkout milestone-024
!python -m pip install -q -U pip
!python -m pip install -q -U "jax[tpu]" flax optax numpy pyarrow datasets huggingface-hub

In [ ]:
"""Download the public tokenizer and shard files from Hugging Face."""

from pathlib import Path
from huggingface_hub import snapshot_download

HF_DATASET_REPO = "marcoshernanz/llm-lab-fineweb-edu-sample10bt-bpe-16384"
LOCAL_DATA_ROOT = Path("/kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384")
LOCAL_TOKENIZER_PATH = LOCAL_DATA_ROOT / "fineweb_edu_sample10bt_bpe_16384.json"

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    local_dir=str(LOCAL_DATA_ROOT),
    allow_patterns=["*.json", "*.npy"],
    token=False,
)

print("local_dataset_root=", LOCAL_DATA_ROOT)
print("local_tokenizer_path=", LOCAL_TOKENIZER_PATH)
print("downloaded_files=")
for path in sorted(LOCAL_DATA_ROOT.iterdir()):
    print("  ", path.name)


In [ ]:
"""Configure the milestone-024 batch-size sweep."""

from pathlib import Path

EXPERIMENT_SCRIPT = "experiments/024_tpu_fineweb_edu_batch_size_sweep.py"
EXPERIMENT_NAME = "024_tpu_fineweb_edu_batch_size_sweep"
ARTIFACTS_ROOT = Path("/kaggle/working/artifacts/experiments")
RUNS_ROOT = ARTIFACTS_ROOT / EXPERIMENT_NAME
BATCH_SIZES = [32, 64, 128, 192, 256]
TRAIN_STEPS = 20000
TRAIN_CHUNK_LENGTH = 100
LEARNING_RATE = 0.05
MAX_TRAIN_SHARDS = 10

COMMON_ARGS = [
    "--token-shard-root",
    "/kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384",
    "--tokenizer-path",
    "/kaggle/working/llm-lab/datasets/fineweb_edu/sample10bt_bpe_16384/fineweb_edu_sample10bt_bpe_16384.json",
    "--artifacts-root",
    str(ARTIFACTS_ROOT),
    "--max-train-shards",
    str(MAX_TRAIN_SHARDS),
    "--train-steps",
    str(TRAIN_STEPS),
    "--train-chunk-length",
    str(TRAIN_CHUNK_LENGTH),
    "--learning-rate",
    str(LEARNING_RATE),
]

print("planned_batch_sizes=", BATCH_SIZES)
print("train_steps=", TRAIN_STEPS)
print("learning_rate=", LEARNING_RATE)
print("artifacts_root=", ARTIFACTS_ROOT)


In [ ]:
"""Run one sweep point with explicit batch-size metadata."""

import os
import shlex
import subprocess


def run_batch_size(batch_size: int) -> None:
    """Run one milestone-024 batch-size configuration."""
    cmd = [
        "python",
        EXPERIMENT_SCRIPT,
        *COMMON_ARGS,
        "--batch-size",
        str(batch_size),
        "--execution-target",
        f"Kaggle TPU v5e-8 batch_size={batch_size}",
    ]
    print("running_command=", shlex.join(cmd))
    env = dict(os.environ)
    env["PYTHONPATH"] = "/kaggle/working/llm-lab"
    subprocess.run(cmd, check=True, cwd="/kaggle/working/llm-lab", env=env)


In [ ]:
"""Run one selected batch size. Change the value and rerun this cell."""

BATCH_SIZE_TO_RUN = 32
run_batch_size(BATCH_SIZE_TO_RUN)


In [ ]:
"""Run the full milestone-024 sweep in sequence."""

for batch_size in BATCH_SIZES:
    run_batch_size(batch_size)


In [ ]:
"""Summarize all saved run metadata for the batch-size sweep."""

import json

records = []
for run_dir in sorted(path for path in RUNS_ROOT.glob("*") if path.is_dir()):
    metadata_path = run_dir / "run_metadata.json"
    if not metadata_path.exists():
        continue
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    records.append(
        {
            "run_dir": run_dir.name,
            "batch_size": metadata["batch_size"],
            "final_train_loss": metadata["final_train_loss"],
            "final_validation_subset_loss": metadata["final_validation_subset_loss"],
            "tokens_per_second": metadata["tokens_per_second"],
        }
    )

records.sort(key=lambda record: record["batch_size"])
if not records:
    raise FileNotFoundError(f"No runs found under {RUNS_ROOT}")

print("batch_size | train_loss | validation_loss | tokens_per_second | run_dir")
for record in records:
    print(
        f"{record['batch_size']:>10} | "
        f"{record['final_train_loss']:.6f} | "
        f"{record['final_validation_subset_loss']:.6f} | "
        f"{record['tokens_per_second']:.3f} | "
        f"{record['run_dir']}"
    )


In [ ]:
"""Inspect one saved run by batch size."""

import json
from IPython.display import SVG, display

BATCH_SIZE_TO_INSPECT = 128
matching_run_dirs = []
for run_dir in sorted(path for path in RUNS_ROOT.glob("*") if path.is_dir()):
    metadata_path = run_dir / "run_metadata.json"
    if not metadata_path.exists():
        continue
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata["batch_size"] == BATCH_SIZE_TO_INSPECT:
        matching_run_dirs.append((run_dir, metadata))

if not matching_run_dirs:
    raise FileNotFoundError(f"No run found for batch_size={BATCH_SIZE_TO_INSPECT}")

run_dir, metadata = matching_run_dirs[-1]
print("run_dir=", run_dir)
display(SVG(filename=str(run_dir / "loss_curve.svg")))
print(json.dumps(metadata, indent=2))
print((run_dir / "sample.txt").read_text(encoding="utf-8"))


In [ ]:
"""Zip the full experiment run directory so Kaggle exposes one download."""

import shutil

OUTPUT_ROOT = Path("/kaggle/working")
archive_base = OUTPUT_ROOT / f"{EXPERIMENT_NAME}_runs"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=str(RUNS_ROOT.parent), base_dir=RUNS_ROOT.name))

print("runs_root=", RUNS_ROOT)
print("archive_path=", archive_path)
print("download_from_kaggle_output_panel=", archive_path.name)
